# FedKDL Kaggle Experiments Runner
Notebook này giúp bạn chạy từng bước của quy trình huấn luyện FedKDL một cách dễ dàng, tránh bị ngắt kết nối và dễ dàng theo dõi log.

### 1. Tải Source Code từ GitHub
Clone repo FedKDL mới nhất từ GitHub và di chuyển vào thư mục làm việc.

In [ ]:
!git clone https://github.com/ngnam1104/FedKDL.git
import os
os.chdir('/kaggle/working/FedKDL')
print('Current Working Directory:', os.getcwd())

### 2. Cài đặt thư viện Cài đặt thư viện và Khởi tạo đường dẫn


In [ ]:
!pip install ultralytics pandas pyyaml torch torchvision
import os, sys
sys.path.append(os.getcwd())

!mkdir -p /kaggle/working/FedKDL/datasets/URPC2020
!ln -s /kaggle/input/datasets/lywang777/urpc2020/URPC2020 /kaggle/working/FedKDL/datasets/URPC2020/URPC2020



### 3. Tạo Topologies và Data Partitions

In [ ]:
!python utils/generate_all_envs.py --n 30 --dataset URPC --m-relays 5 --alphas 1.0

### 3.5. Tạo Proxy Dataset (15% URPC)
Trích xuất 15% dữ liệu URPC để tạo tập proxy dùng cho việc Warmup (trước FL) và Knowledge Distillation (tại Gateway).

In [ ]:
import os, random, yaml
from pathlib import Path

orig_yaml = "/kaggle/input/datasets/lywang777/urpc2020/URPC2020/data.yaml"
proxy_yaml = "datasets/URPC2020_proxy.yaml"

if not os.path.exists(orig_yaml):
    print(f"Không tìm thấy {orig_yaml}")
else:
    with open(orig_yaml, "r") as f:
        config = yaml.safe_load(f)

    # Theo cấu trúc của dataset URPC2020 trên Kaggle
    dataset_root = "/kaggle/input/datasets/lywang777/urpc2020/URPC2020"
    train_images_dir = Path(f"{dataset_root}/train/images")
    train_labels_dir = Path(f"{dataset_root}/train/labels")

    if train_images_dir.exists() and train_labels_dir.exists():
        all_imgs = sorted(list(train_images_dir.glob("*.jpg")))
        HABITAT_TO_URPC = {0: 2, 1: 1, 2: 3, 3: 0}
        URPC_TO_HABITAT = {v: k for k, v in HABITAT_TO_URPC.items()}

        imgs_by_habitat = {h: [] for h in range(4)}
        imgs_noclass = []

        print("Đang phân loại ảnh theo Habitat...")
        for img_path in all_imgs:
            lbl_path = train_labels_dir / (img_path.stem + ".txt")
            counts = {c: 0 for c in range(4)}
            try:
                if lbl_path.exists():
                    with open(lbl_path, "r") as lf:
                        for line in lf:
                            parts = line.strip().split()
                            if parts:
                                cls_id = int(parts[0])
                                if cls_id in counts:
                                    counts[cls_id] += 1
                    if sum(counts.values()) == 0:
                        imgs_noclass.append(img_path)
                    else:
                        dominant = max(counts, key=counts.get)
                        habitat = URPC_TO_HABITAT.get(dominant, 0)
                        imgs_by_habitat[habitat].append(img_path)
                else:
                    imgs_noclass.append(img_path)
            except Exception:
                imgs_noclass.append(img_path)

        for i, img_path in enumerate(imgs_noclass):
            imgs_by_habitat[i % 4].append(img_path)

        random.seed(1104)
        proxy_imgs = []
        print("\n[Data Partitioning] Tách 15% Proxy Data từ các Habitat:")
        for h in range(4):
            pool = imgs_by_habitat[h]
            old_size = len(pool)
            proxy_for_h = int(old_size * 0.15)
            sampled = random.sample(pool, proxy_for_h)
            proxy_imgs.extend(sampled)
            print(f"    - Habitat {h}: Tổng {old_size} ảnh -> Lấy {proxy_for_h} ảnh cho KD")

        proxy_txt = Path("datasets/URPC2020/proxy_train.txt")
        proxy_txt.parent.mkdir(parents=True, exist_ok=True)
        with open(proxy_txt, "w") as f:
            for img in proxy_imgs:
                f.write(f"{img.absolute()}\n")

        # Fix absolute paths from original data.yaml
        config["path"] = dataset_root
        config["train"] = str(proxy_txt.absolute())
        config["val"] = "valid/images"
        config["test"] = "test/images"

        with open(proxy_yaml, "w") as f:
            yaml.dump(config, f)
        print(f"\n=> Đã tạo {proxy_yaml} với tổng {len(proxy_imgs)} ảnh proxy (15%).")
    else:
        print("Chưa tìm thấy thư mục ảnh hoặc nhãn. Hãy kiểm tra lại đường dẫn dataset.")


### 4.1. Pre-train Centralized Full-Finetune (Upper Bound)
Chạy Centralized Full Finetune trên toàn bộ dữ liệu để có baseline Upper Bound cho cấu trúc gốc không dùng LoRA.

In [ ]:
!python scripts/fedkdl/train_student_warmup.py --mode centralized_full
# Nếu bị ngắt giữa chừng, đổi thành lệnh dưới để chạy tiếp:
# !python scripts/fedkdl/train_student_warmup.py --mode centralized_full --resume

### 4.2. Pre-train Centralized LoRA
Chạy Centralized LoRA trên toàn bộ dữ liệu để so sánh hiệu năng trực tiếp với Full Finetune.

In [ ]:
!python scripts/fedkdl/train_student_warmup.py --mode centralized_lora
# Nếu bị ngắt giữa chừng, đổi thành lệnh dưới để chạy tiếp:
# !python scripts/fedkdl/train_student_warmup.py --mode centralized_lora --resume

### 5. Khởi chạy thuật toán FedKDL (Main Baseline)
**Giải đáp về Log:** Trong lệnh dưới, toán tử `2>&1 | tee ...` sẽ vừa in log ra giao diện Kaggle Notebook để bạn xem trực tiếp, VỪA ghi song song toàn bộ log vào file `.log`. Kể cả khi Notebook của bạn bị reload/treo trình duyệt, file log vật lý trên hệ thống Kaggle vẫn được lưu bình thường không sót chữ nào!

In [ ]:
!pip uninstall -y ray

In [ ]:
!mkdir -p results/logs_kdl results/train_logs/kdl

!python main_trainer_od.py \
    --topo environments/2d/topo/N_30/topo_N30_seed1104.pkl \
    --data environments/2d/data/URPC/N_30/data_N30_URPC_a1p0_seed1104.pkl \
    --baseline fedkdl \
    --rounds 60 \
    --out-dir results/logs_kdl \
    --log-dir results/train_logs/kdl \
    2>&1 | tee results/train_logs/kdl/kaggle_fedkdl_raw.log

### 6. Khởi chạy các Kịch bản So sánh (Ablation)
Bạn có thể đổi `baseline` thành `fedkdl_selective`, `fedprox_kdl`, `fedkd` tùy nhu cầu để chạy các thử nghiệm khác.

In [ ]:
!python main_trainer_od.py \
    --topo environments/2d/topo/N_30/topo_N30_seed1104.pkl \
    --data environments/2d/data/URPC/N_30/data_N30_URPC_a1p0_seed1104.pkl \
    --baseline fedkd \
    --rounds 60 \
    --out-dir results/logs_kdl \
    --log-dir results/train_logs/kdl \
    2>&1 | tee results/train_logs/kdl/kaggle_fedkd_raw.log